# Plot2Code Dataset Generation

This notebook generates a dataset of matplotlib plots and their corresponding code for training a vision-language model to generate plotting code from images.

In [1]:
import time
import subprocess, warnings, json, time, matplotlib
warnings.filterwarnings("ignore")
matplotlib.use("Agg")
from pathlib import Path
root_dir = Path().resolve()

from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import random
random.seed(42)
from augment import augment_dataset, get_all_py_files, should_skip_code
import shutil
from common_utils import train_gallery_dir, train_dataset_file, test_gallery_dir, test_dataset_file, remove_comments, check_png_quality_from_json, check_png_quality
from vegalite_utils import parse_vegalite_spec, matplotlib_code_to_vegalite, ensure_sample_vegalite_spec, max_workers

# -------------------------
# Step 1: Setup base paths
# -------------------------
DATA_DIR = root_dir / "data"
MPL_DIR = DATA_DIR / "matplotlib"
SBN_DIR = DATA_DIR / "seaborn"

TRAIN_DIR = root_dir / "train_data"
TEST_DIR = root_dir / "test_data"

BASE_PATHS = [root_dir / MPL_DIR, root_dir / SBN_DIR]

In [ ]:
# from common_utils import train_dataset_file, test_dataset_file

# files = [train_dataset_file, test_dataset_file]
# for file in files:
#     with open(file, "r") as f:
#         lines = f.readlines()

#     lines_to_check = []
#     for line in lines:
#         if "vega_spec" not in line:
#             lines_to_check.append(line)

#     with open(f"{file.stem}_temp.jsonl", "w") as fw:
#         fw.write("".join(lines_to_check))

In [2]:
if train_dataset_file.exists():
    print(f"{train_dataset_file} exists. Removing it to create a fresh one.")
    train_dataset_file.unlink()
if test_dataset_file.exists():
	print(f"{test_dataset_file} exists. Removing it to create a fresh one.")
	test_dataset_file.unlink()      
if train_gallery_dir.exists():
	print(f"{train_gallery_dir} exists. Removing it to create a fresh one.")
	shutil.rmtree(train_gallery_dir)
if test_gallery_dir.exists():
	print(f"{test_gallery_dir} exists. Removing it to create a fresh one.")
	shutil.rmtree(test_gallery_dir)

In [3]:
def clone_repo(repo_url, target_path):
	"""Clone repo if not already present"""
	if target_path.exists():
		print(f"Repo already exists: {target_path}")
		print(f"Removing existing folder: {target_path}")
		shutil.rmtree(target_path)

	print(f"Cloning {repo_url}...")
	subprocess.run(["git", "clone", repo_url, str(target_path)], check=True)


def keep_only_required_folder(base_path, folder_to_keep):
	"""Delete everything except the required folder"""

	for item in base_path.iterdir():
		if item.name != folder_to_keep:
			if item.is_dir():
				try:
					shutil.rmtree(item)
				except PermissionError:
					print(f"Permission denied: {item}")
			else:
				item.unlink()

def copy_with_structure(src_path, base_path, target_root):

	with open(src_path, 'r', encoding='utf-8') as f:
		content = f.read()

	if not should_skip_code(content):
		relative_path = src_path.relative_to(base_path)
		dest_path = target_root / base_path.name / relative_path

		dest_path.parent.mkdir(parents=True, exist_ok=True)
		shutil.copy2(src_path, dest_path)

		cleaned_content = remove_comments(content)
		with open(dest_path, 'w', encoding='utf-8') as f:
			f.write(cleaned_content)
	else:
		print(f"Skipping {src_path} due to should_skip_code criteria.")


def create_train_test_data():
	random.seed(42)

	# -------------------------
	# Step 2: Clone repos
	# -------------------------
	if not DATA_DIR.exists():
		DATA_DIR.mkdir(exist_ok=True)
		clone_repo("https://github.com/matplotlib/matplotlib.git", MPL_DIR)
		time.sleep(10)  # Just to ensure the first clone is fully done before starting the next one
		print("Cleaning matplotlib repo...")
		keep_only_required_folder(MPL_DIR, "galleries")

		clone_repo("https://github.com/mwaskom/seaborn.git", SBN_DIR)
		time.sleep(10)  # Ensure the second clone is fully done
		print("Cleaning seaborn repo...")
		keep_only_required_folder(SBN_DIR, "examples")

	# -------------------------
	# Step 3: Collect files
	# -------------------------
	all_files = []
	for base in BASE_PATHS:
		files = get_all_py_files(base)
		all_files.extend([(f, base) for f in files])

	print(f"Total files found: {len(all_files)}")

	# -------------------------
	# Step 4: Split (60/40)
	# -------------------------
	random.shuffle(all_files)
	split_idx = int(0.6 * len(all_files))

	train_files = all_files[:split_idx]
	test_files = all_files[split_idx:]

	print(f"Train: {len(train_files)}, Test: {len(test_files)}")

	# -------------------------
	# Step 5: Copy files
	# -------------------------
	for file_path, base in train_files:
		copy_with_structure(file_path, base, TRAIN_DIR)

	for file_path, base in test_files:
		copy_with_structure(file_path, base, TEST_DIR)

	print("✅ Done! Dataset ready inside 'data/' folder")

if TRAIN_DIR.exists():
	print(f"{TRAIN_DIR} already exists. Removing it to create a fresh dataset.")
	shutil.rmtree(TRAIN_DIR)
TRAIN_DIR.mkdir(exist_ok=True)

if TEST_DIR.exists():
	print(f"{TEST_DIR} already exists. Removing it to create a fresh dataset.")
	shutil.rmtree(TEST_DIR)
TEST_DIR.mkdir(exist_ok=True)

create_train_test_data()

Total files found: 633
Train: 379, Test: 254
Skipping /home/em-plot/plot2code/Dhruv/plot2code/data/matplotlib/galleries/examples/subplots_axes_and_figures/figure_title.py due to should_skip_code criteria.
Skipping /home/em-plot/plot2code/Dhruv/plot2code/data/matplotlib/galleries/examples/misc/font_indexing.py due to should_skip_code criteria.
Skipping /home/em-plot/plot2code/Dhruv/plot2code/data/matplotlib/galleries/examples/mplot3d/contour3d_3.py due to should_skip_code criteria.
Skipping /home/em-plot/plot2code/Dhruv/plot2code/data/seaborn/examples/scatterplot_matrix.py due to should_skip_code criteria.
Skipping /home/em-plot/plot2code/Dhruv/plot2code/data/matplotlib/galleries/users_explain/axes/legend_guide.py due to should_skip_code criteria.
Skipping /home/em-plot/plot2code/Dhruv/plot2code/data/matplotlib/galleries/examples/mplot3d/voxels_torus.py due to should_skip_code criteria.
Skipping /home/em-plot/plot2code/Dhruv/plot2code/data/matplotlib/galleries/examples/lines_bars_and_ma

## Step 2: Filter Example Categories

Keep only specific plot categories and remove others to focus on relevant plot types.

In [4]:
train_examples = [f for f in TRAIN_DIR.rglob("*.py")]
print(f"Found {len(train_examples)} examples under: {TRAIN_DIR}")

Found 145 examples under: /home/em-plot/plot2code/Dhruv/plot2code/train_data


In [5]:
test_examples = [f for f in TEST_DIR.rglob("*.py")]
print(f"Found {len(test_examples)} examples under: {TEST_DIR}")

Found 85 examples under: /home/em-plot/plot2code/Dhruv/plot2code/test_data


## Step 3: Generate Dataset

Execute matplotlib example scripts and save the resulting plots with their source code.

In [6]:
failed_examples = {}


def run_example(f, gallery_dir):
    out_png = gallery_dir / (f.stem + ".png")
    code = f.read_text(encoding="utf-8", errors="ignore")

    cmd = f"""
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, numpy as np
exec(open(r'{f}').read(), {{'plt': plt, 'np': np}})
plt.savefig(r'{out_png}', bbox_inches='tight', dpi=150)
"""
    process = subprocess.Popen(
        ["python", "-c", cmd],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    try:
        process.wait(timeout=30)
    except:
        print(f"\nTimeout Expired: {f.name} - {process.returncode}")

    image_name = out_png.name

    if process.returncode == 0 and out_png.exists() and check_png_quality(image_name, out_png):
        return str(out_png), code
    else:
        # print(f"PNG quality check failed for: {out_png} (size: {out_png.stat().st_size} bytes)")    
        if f.parent.stem not in failed_examples.keys():
            failed_examples[f.parent.stem] = []
        if f.stem not in failed_examples[f.parent.stem]:
            failed_examples[f.parent.stem].append(f.stem)

In [7]:
def create_json_data(gallery_dir, dataset_file, examples):
	start_para = time.time()
	gallery_dir.mkdir(exist_ok=True, parents=True)

	print(f"Output dataset: {dataset_file}")
	print(f"Found {len(examples)} example scripts to process")

	# Launch parallel subprocesses using threads
	results = []

	print("\nWorkers:", max_workers)
	with ThreadPoolExecutor(max_workers=max_workers) as executor:
		futures = {
			executor.submit(run_example, f, gallery_dir): f for f in examples
		}
		for future in tqdm(
			as_completed(futures), total=len(futures), desc="Launching jobs"
		):
			results.append(future.result())

	# Save JSONL
	with open(dataset_file, "w", encoding="utf-8") as f:
		for d in results:
			if d is not None:
				f.write(json.dumps({"image": d[0], "code": d[1]}) + "\n")

	print(f"✅ Generated {len(results)} valid (image, code) pairs")
	print(f"📁 Saved to {dataset_file.resolve()}")
	print(f"⏱️ Total time: {time.time() - start_para:.1f} sec")

	check_png_quality_from_json(dataset_file)

	# Print length of dataset_file
	with open(dataset_file, 'r', encoding='utf-8') as f:
		lines = f.readlines()
	print(f"Total valid examples in {dataset_file.name}: {len(lines)}")


In [8]:
# Create Training data
create_json_data(train_gallery_dir, train_dataset_file, examples=train_examples)

Output dataset: /home/em-plot/plot2code/Dhruv/plot2code/plot2code_train.jsonl
Found 145 example scripts to process

Workers: 14


Launching jobs: 100%|██████████| 145/145 [00:18<00:00,  7.91it/s]


✅ Generated 145 valid (image, code) pairs
📁 Saved to /home/em-plot/plot2code/Dhruv/plot2code/plot2code_train.jsonl
⏱️ Total time: 18.4 sec
Total PNG files checked: 143
143 PNG files exist.
Total valid examples in plot2code_train.jsonl: 143


In [9]:
# Create Testing data
create_json_data(test_gallery_dir, test_dataset_file, examples=test_examples)

Output dataset: /home/em-plot/plot2code/Dhruv/plot2code/plot2code_test.jsonl
Found 85 example scripts to process

Workers: 14


Launching jobs:   0%|          | 0/85 [00:00<?, ?it/s]

Launching jobs: 100%|██████████| 85/85 [00:09<00:00,  8.53it/s]


✅ Generated 85 valid (image, code) pairs
📁 Saved to /home/em-plot/plot2code/Dhruv/plot2code/plot2code_test.jsonl
⏱️ Total time: 10.0 sec
Total PNG files checked: 82
82 PNG files exist.
Total valid examples in plot2code_test.jsonl: 82


In [10]:
count = 0
for fail_key, fail_val in failed_examples.items():
	count += len(fail_val)
print(f"Failed examples: {count} scripts in {len(failed_examples)} categories")

Failed examples: 5 scripts in 5 categories


In [11]:
# Augment Dataset

augment_dataset(train_gallery_dir, TRAIN_DIR, TRAIN_DIR / "augmented_files", train_dataset_file)


Processing batch 1/4


Batch progress:   2%|▏         | 3/200 [00:05<05:00,  1.53s/it]

No axes in figure for image: mathtext_aug_0_0.png

No plotted content for image: mathtext_aug_0_1.png
No axes in figure for image: mathtext_aug_0_2.png

No plotted content for image: mathtext_aug_1_0.png

No plotted content for image: mathtext_aug_1_1.png

No plotted content for image: mathtext_aug_1_2.png

No plotted content for image: mathtext_aug_3_0.png
No axes in figure for image: mathtext_aug_3_1.png

No plotted content for image: mathtext_aug_3_2.png

No plotted content for image: mathtext_aug_4_0.png

No plotted content for image: mathtext_aug_4_1.png
No axes in figure for image: mathtext_aug_4_2.png


Batch progress:   2%|▎         | 5/200 [00:05<02:31,  1.29it/s]


No plotted content for image: mathtext_aug_2_0.png

No plotted content for image: mathtext_aug_2_1.png
No axes in figure for image: mathtext_aug_2_2.png


...


Processing batch 4/4


Batch progress:  79%|███████▉  | 99/125 [00:14<00:02, 12.04it/s]

using 52 of 1396 visible points
using 52 of 1396 visible points
using 52 of 1396 visible points


Batch progress: 100%|██████████| 125/125 [00:16<00:00,  7.45it/s]



Total generated samples: 1727


In [12]:
# Move irrelevant files to a separate folder
irrelevant_dir = root_dir / "irrelevant_files"
irrelevant_dir.mkdir(exist_ok=True)

irrelevant_files = [
    "image.svg",
    "multipage_pdf.pdf",
    "scatter.svg",
    "test_rasterization.eps",
    "test_rasterization.pdf",
    "test_rasterization.svg",
    "test.bmp",
    "test.png",
    "svg_filter_line.svg",
    "svg_histogram.svg",
    "svg_tooltip.svg"
]

for file in irrelevant_files:
    try:
        shutil.move(str(root_dir / file), irrelevant_dir / file)
    except FileNotFoundError:
        print(f"File not found, skipping move: {file}")

# Delete irrelevant_dir
shutil.rmtree(irrelevant_dir)

File not found, skipping move: test.bmp
File not found, skipping move: svg_filter_line.svg
File not found, skipping move: svg_histogram.svg
File not found, skipping move: svg_tooltip.svg


In [13]:
print("\nFinal Check of the training dataset PNG files...")
check_png_quality_from_json(train_dataset_file)


Final Check of the training dataset PNG files...
Total PNG files checked: 1870
1870 PNG files exist.


In [14]:
print("\nFinal Check of the test dataset PNG files...")
check_png_quality_from_json(test_dataset_file)


Final Check of the test dataset PNG files...
Total PNG files checked: 82
82 PNG files exist.


In [15]:
def add_vega_spec_to_jsonl(jsonl_file):
    """Populate `vega_spec` using multiple strategies in order:
       1) parse explicit Vega-Lite JSON
       2) convert matplotlib code -> Vega-Lite
       3) programmatically generate via ensure_sample_vegalite_spec
       Writes back only when a valid non-empty spec is produced.
    """
    temp_file = jsonl_file.with_suffix('.temp')
    failures = []
    converted = 0

    with open(jsonl_file, 'r', encoding='utf-8') as infile, open(temp_file, 'w', encoding='utf-8') as outfile:
        for line in infile:
            data = json.loads(line)
            code = data.get("code", "")
            spec = None

            # 1) parse embedded Vega-Lite JSON (if present)
            try:
                spec = parse_vegalite_spec(code)
            except Exception:
                spec = None

            # 2) convert matplotlib code -> vega
            if spec is None and code:
                try:
                    spec = matplotlib_code_to_vegalite(code)
                except Exception:
                    spec = None

            # 3) try programmatic generation from sample
            if spec is None:
                try:
                    sample = {"code": code, "image": data.get("image")}
                    spec = ensure_sample_vegalite_spec(sample, save=False)
                except Exception:
                    spec = None

            # Normalize & validate spec before writing
            if spec:
                # optional: canonical_json_dumps(spec) if you want stable, pretty JSON
                data["vega_spec"] = spec
                converted += 1
            else:
                # keep it absent rather than writing empty {} — easier to detect failures
                failures.append(data.get("image"))

            json.dump(data, outfile)
            outfile.write("\n")

    temp_file.replace(jsonl_file)
    print(f"add_vega_spec_to_jsonl: converted={converted}, failures={len(failures)}")
    if failures:
        print("Examples failed to produce vega_spec (showing up to 10):", failures[:10])

In [16]:
add_vega_spec_to_jsonl(train_dataset_file)
add_vega_spec_to_jsonl(test_dataset_file)

add_vega_spec_to_jsonl: converted=1009, failures=861
Examples failed to produce vega_spec (showing up to 10): ['/home/em-plot/plot2code/Dhruv/plot2code/plot_gallery_train/quiver.png', '/home/em-plot/plot2code/Dhruv/plot2code/plot_gallery_train/triplot.png', '/home/em-plot/plot2code/Dhruv/plot2code/plot_gallery_train/eventplot.png', '/home/em-plot/plot2code/Dhruv/plot2code/plot_gallery_train/mathtext.png', '/home/em-plot/plot2code/Dhruv/plot2code/plot_gallery_train/text_props.png', '/home/em-plot/plot2code/Dhruv/plot2code/plot_gallery_train/barbs.png', '/home/em-plot/plot2code/Dhruv/plot2code/plot_gallery_train/streamplot.png', '/home/em-plot/plot2code/Dhruv/plot2code/plot_gallery_train/dark_background.png', '/home/em-plot/plot2code/Dhruv/plot2code/plot_gallery_train/trigradient_demo.png', '/home/em-plot/plot2code/Dhruv/plot2code/plot_gallery_train/anatomy.png']
add_vega_spec_to_jsonl: converted=39, failures=43
Examples failed to produce vega_spec (showing up to 10): ['/home/em-plot/plo